# Blockfish Colab Auto-Polling Worker

**One-time setup (zero manual clicking afterwards!):**
1. In Colab left sidebar, click **Secrets (key icon)** and add `GH_TOKEN` (a GitHub Personal Access Token with `repo` scope), then toggle **Notebook access** ON. *(If not set, Cell 2 will prompt once.)*
2. Recommended runtime: **High-RAM CPU** (or **L4 / T4 / A100** when we run GPU training) so you get 8–12+ fast CPU cores.
3. Click **Runtime → Run all** ONCE. Cell 3 will poll GitHub every 15s, automatically run any new `scripts/colab_cmd.sh` pushed by the agent, and push `results/colab_latest.log` + `models/*.txt` back with `[skip ci]`.

In [ ]:
# Cell 1: Check hardware & install Rust toolchain if needed
!nproc && free -h
!nvidia-smi 2>/dev/null || echo 'CPU-only runtime'
import os, subprocess
cargo_bin = os.path.expanduser('~/.cargo/bin/cargo')
if not os.path.exists(cargo_bin):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version && cargo --version

In [ ]:
# Cell 2: Clone / sync jush0147/blockfish using GH_TOKEN
import os, getpass

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

if not GH_TOKEN:
    GH_TOKEN = getpass.getpass('Paste GitHub PAT (repo scope): ')

REPO_DIR = '/content/blockfish'
BRANCH = 'rescue/independent-validation-20261116'
REMOTE_URL = f'https://{GH_TOKEN}@github.com/jush0147/blockfish.git'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REMOTE_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git remote set-url origin {REMOTE_URL}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

%cd {REPO_DIR}
!git config user.name 'colab-worker[bot]'
!git config user.email 'colab-worker@users.noreply.github.com'
!git status -s && echo 'Repo ready!'

In [ ]:
# Cell 3: Start Auto-Polling Worker (runs continuously, zero manual clicks needed!)
%cd /content/blockfish
!chmod +x scripts/colab_worker.sh scripts/colab_cmd.sh
!bash scripts/colab_worker.sh